# Layerwise steady-state and coupled-criterion demo

One shared skier setup, then:

1. **Full profile** — weak layer at the base of the entire slab.
2. **Layerwise sweep** — truncate the slab to each weak-layer depth (20 cm spacing on a 70 cm profile).

Each run reports minimum force, coupled criterion, maximal stress, crack length, and hybrid steady state.

In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from __future__ import annotations

import copy
from dataclasses import dataclass

import numpy as np
import pandas as pd
from pydantic import BaseModel

from weac.analysis.criteria_evaluator import CriteriaEvaluator
from weac.components import (
    Config,
    CriteriaConfig,
    Layer,
    ModelInput,
    ScenarioConfig,
    Segment,
    WeakLayer,
)
from weac.core.system_model import SystemModel

## Result types (API-shaped)

In [11]:
class SteadyStateTensileMetrics(BaseModel):
    critical_cut_length: float
    cut_direction_winner: str
    converged: bool
    max_Sxx_norm: float | None = None


class SteadyStateErrMetrics(BaseModel):
    energy_release_rate: float
    cut_length: float
    cut_direction_winner: str
    converged: bool


class SteadyStateMetrics(BaseModel):
    tensile: SteadyStateTensileMetrics
    err: SteadyStateErrMetrics
    phi: float
    converged: bool


class RawAnalysisMetrics(BaseModel):
    """Raw metrics for a single weak-layer depth."""

    wl_depth: float
    min_force_critical_weight: float
    coupled_critical_weight: float
    max_stress: float
    crack_length: float
    steady_state: SteadyStateMetrics


@dataclass
class _DepthTask:
    wl_depth: float
    layers: list[Layer]
    weak_layer: WeakLayer


def _build_depth_tasks(
    *,
    layers: list[Layer],
    weak_layer: WeakLayer,
    wl_spacing: int,
) -> list[_DepthTask]:
    """One task per weak-layer depth (truncate slab at ``wl_depth``)."""
    layers_copy = copy.deepcopy(layers)
    heights = np.cumsum([layer.h for layer in layers_copy])
    wl_depths = np.arange(wl_spacing, heights[-1], wl_spacing).tolist()
    wl_depths.append(float(heights[-1]))

    tasks: list[_DepthTask] = []
    for wl_depth in wl_depths:
        mask = heights <= wl_depth
        new_layers = [layer for layer, keep in zip(layers_copy, mask) if keep]

        depth = float(np.sum([layer.h for layer in new_layers])) if new_layers else 0.0
        if depth < wl_depth:
            additional_layer = copy.deepcopy(
                layers_copy[len(new_layers) if new_layers else 0]
            )
            additional_layer.h = wl_depth - depth
            new_layers.append(additional_layer)

        tasks.append(
            _DepthTask(
                wl_depth=float(wl_depth),
                layers=new_layers,
                weak_layer=copy.deepcopy(weak_layer),
            )
        )
    return tasks

## Shared setup (70 cm slab, skier load)

In [12]:
PHI = 30.0
WL_SPACING_MM = 200  # 20 cm

full_layers = [
    Layer(rho=280, h=400),
    Layer(rho=220, h=300),
]
assert sum(layer.h for layer in full_layers) == 700

weak_layer = WeakLayer(rho=150, h=20, E=0.25)

scenario_config = ScenarioConfig(system_type="skier", phi=PHI)
segments = [
    Segment(length=5000, has_foundation=True, m=0),
    Segment(length=0, has_foundation=False, m=75),
    Segment(length=0, has_foundation=False, m=0),
    Segment(length=5000, has_foundation=False, m=0),
]

criteria_config = CriteriaConfig(
    stress_envelope_method="adam_unpublished",
    scaling_factor=1,
    order_of_magnitude=1,
)
criteria_evaluator = CriteriaEvaluator(criteria_config)
system_config = Config(touchdown=True)


def make_system(*, layers: list[Layer], weak_layer: WeakLayer) -> SystemModel:
    return SystemModel(
        model_input=ModelInput(
            scenario_config=scenario_config,
            layers=layers,
            segments=copy.deepcopy(segments),
            weak_layer=weak_layer,
        ),
        config=system_config,
    )


def run_analysis(*, layers: list[Layer], weak_layer: WeakLayer, wl_depth: float) -> RawAnalysisMetrics:
    system = make_system(layers=layers, weak_layer=weak_layer)

    min_force = criteria_evaluator.find_minimum_force(system)
    coupled = criteria_evaluator.evaluate_coupled_criterion(system)
    ss = criteria_evaluator.evaluate_SteadyState(system)

    max_stress = float(coupled.max_dist_stress)

    tensile_stress = ss.tensile.maximal_stress_result
    max_sxx = (
        float(tensile_stress.max_Sxx_norm)
        if tensile_stress is not None
        else None
    )

    steady = SteadyStateMetrics(
        tensile=SteadyStateTensileMetrics(
            critical_cut_length=float(ss.tensile.critical_cut_length),
            cut_direction_winner=str(ss.tensile.cut_direction_winner),
            converged=bool(ss.tensile.converged),
            max_Sxx_norm=max_sxx,
        ),
        err=SteadyStateErrMetrics(
            energy_release_rate=float(ss.err.energy_release_rate),
            cut_length=float(ss.err.cut_length),
            cut_direction_winner=str(ss.err.cut_direction_winner),
            converged=bool(ss.err.converged),
        ),
        phi=float(ss.phi),
        converged=bool(ss.converged),
    )

    return RawAnalysisMetrics(
        wl_depth=float(wl_depth),
        min_force_critical_weight=float(min_force.critical_skier_weight),
        coupled_critical_weight=float(coupled.critical_skier_weight),
        max_stress=max_stress,
        crack_length=float(coupled.crack_length),
        steady_state=steady,
    )


def print_metrics(label: str, metrics: RawAnalysisMetrics) -> None:
    ss = metrics.steady_state
    print(f"\n{'=' * 60}")
    print(label)
    print(f"{'=' * 60}")
    print(f"Weak-layer depth (mm):     {metrics.wl_depth:.1f}")
    print(f"Min force weight (kg):     {metrics.min_force_critical_weight:.2f}")
    print(f"Coupled crit. weight (kg): {metrics.coupled_critical_weight:.2f}")
    print(f"Max stress envelope dist:  {metrics.max_stress:.4f}")
    print(f"Coupled crack length (mm): {metrics.crack_length:.1f}")
    print(f"Steady state converged:    {ss.converged}  (phi={ss.phi:.1f}°)")
    print(
        f"  Tensile L_crit: {ss.tensile.critical_cut_length:.1f} mm "
        f"({ss.tensile.cut_direction_winner})"
    )
    print(
        f"  ERR G: {ss.err.energy_release_rate:.3f} J/m², "
        f"cut {ss.err.cut_length:.1f} mm ({ss.err.cut_direction_winner})"
    )

## 1) Full profile (weak layer at slab base)

In [13]:
full_depth = float(sum(layer.h for layer in full_layers))
full_metrics = run_analysis(
    layers=full_layers,
    weak_layer=weak_layer,
    wl_depth=full_depth,
)
print_metrics("Full 70 cm profile", full_metrics)


Full 70 cm profile
Weak-layer depth (mm):     700.0
Min force weight (kg):     541.93
Coupled crit. weight (kg): 544.64
Max stress envelope dist:  1.0101
Coupled crack length (mm): 1.0
Steady state converged:    True  (phi=30.0°)
  Tensile L_crit: 410.3 mm (upslope)
  ERR G: 6.592 J/m², cut 1250.0 mm (upslope)


## 2) Layerwise sweep (20 cm weak-layer spacing)

In [14]:
depth_tasks = _build_depth_tasks(
    layers=full_layers,
    weak_layer=weak_layer,
    wl_spacing=WL_SPACING_MM,
)
print(f"Depths (mm): {[t.wl_depth for t in depth_tasks]}")

layerwise_results: list[RawAnalysisMetrics] = []
for task in depth_tasks:
    layerwise_results.append(
        run_analysis(
            layers=task.layers,
            weak_layer=task.weak_layer,
            wl_depth=task.wl_depth,
        )
    )

Depths (mm): [200.0, 400.0, 600.0, 700.0]


In [15]:
rows = []
for m in layerwise_results:
    rows.append(
        {
            "wl_depth_mm": m.wl_depth,
            "min_force_kg": m.min_force_critical_weight,
            "coupled_kg": m.coupled_critical_weight,
            "max_stress": m.max_stress,
            "crack_mm": m.crack_length,
            "ss_ok": m.steady_state.converged,
            "L_crit_mm": m.steady_state.tensile.critical_cut_length,
            "ERR_J_m2": m.steady_state.err.energy_release_rate,
        }
    )

df = pd.DataFrame(rows).sort_values("wl_depth_mm").reset_index(drop=True)
display(df.round({
    "wl_depth_mm": 1,
    "min_force_kg": 2,
    "coupled_kg": 2,
    "max_stress": 4,
    "crack_mm": 1,
    "L_crit_mm": 1,
    "ERR_J_m2": 3,
}))

,wl_depth_mm,min_force_kg,coupled_kg,max_stress,crack_mm,ss_ok,L_crit_mm,ERR_J_m2
0,200.0,337.88,339.57,1.0113,1.0,True,283.9,2.077
1,400.0,488.59,491.03,1.0107,1.0,True,363.3,4.166
2,600.0,526.53,529.16,1.0104,1.0,True,386.1,5.770
3,700.0,541.93,544.64,1.0101,1.0,True,410.3,6.592
